In [3]:
#Re-train and predict all data.
import os, json
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

# Connect to database
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DB_PATH  = os.path.join(BASE_DIR, "data", "raw", "financials.db")
engine   = create_engine(f"sqlite:///{DB_PATH}")

# Load features
df = pd.read_sql("SELECT * FROM features", engine)

# Define feature columns
RATIO_COLS = [
    "current_ratio", "quick_ratio", "cash_ratio",
    "debt_to_equity", "interest_coverage", "debt_to_ebitda",
    "gross_margin", "ebitda_margin", "net_margin",
    "roe", "roa", "operating_cf_ratio", "cf_to_debt", "fcf_margin"
]
DELTA_COLS   = [f"delta_{c}"  for c in RATIO_COLS]
ZSCORE_COLS  = [f"zscore_{c}" for c in RATIO_COLS]
FEATURE_COLS = RATIO_COLS + DELTA_COLS + ZSCORE_COLS

X = df[FEATURE_COLS]
y = df["health_score"] - 1

# Impute, split, train
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, stratify=y, random_state=42
)

model = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                      subsample=0.8, eval_metric="mlogloss", random_state=42)
model.fit(X_train, y_train)

# Predict on FULL dataset
df["predicted_score"] = model.predict(X_imputed) + 1  # back to 1-5 scale
df["distress_prob"]   = model.predict_proba(X_imputed)[:, 0]

print("Predictions complete!")
print(df[["company_name", "date", "health_score", "predicted_score", "distress_prob"]].head(8))

Predictions complete!
     company_name                        date  health_score  predicted_score  \
0  Anglo American  2022-12-31 00:00:00.000000             5                5   
1  Anglo American  2023-12-31 00:00:00.000000             5                5   
2  Anglo American  2024-12-31 00:00:00.000000             1                1   
3  Anglo American  2025-12-31 00:00:00.000000             4                4   
4       AngloGold  2021-12-31 00:00:00.000000             5                5   
5       AngloGold  2022-12-31 00:00:00.000000             5                5   
6       AngloGold  2023-12-31 00:00:00.000000             1                1   
7       AngloGold  2024-12-31 00:00:00.000000             5                5   

   distress_prob  
0       0.000709  
1       0.010228  
2       0.925038  
3       0.016013  
4       0.000673  
5       0.001264  
6       0.907126  
7       0.000891  


In [4]:
#Building and exporting JSON for React.
LABEL_MAP = {1: "Distressed", 2: "Weak", 3: "Neutral", 4: "Good", 5: "Excellent"}

df["score_label"] = df["predicted_score"].map(LABEL_MAP)
df["year"]        = pd.to_datetime(df["date"]).dt.year

# 1. All records for scatter + trend charts
records = df[[
    "ticker", "company_name", "sector", "year",
    "predicted_score", "score_label", "distress_prob",
    "debt_to_equity", "interest_coverage", "ebitda_margin",
    "gross_margin", "roe", "cf_to_debt", "debt_to_ebitda"
]].round(3).to_dict(orient="records")

# 2. Sector averages for bar chart
sector_avg = (df.groupby("sector")["predicted_score"]
              .mean().round(2).reset_index()
              .rename(columns={"predicted_score": "avg_score"}))
sector_data = sector_avg.to_dict(orient="records")

# 3. Latest score per company for top-10 distress chart
latest   = df.sort_values("year").groupby("ticker").last().reset_index()
top_risk = (latest.sort_values("distress_prob", ascending=False)
            .head(10)[["company_name", "sector", "predicted_score", "score_label", "distress_prob"]]
            .round(3).to_dict(orient="records"))

# 4. Feature importance
feat_imp     = pd.Series(model.feature_importances_, index=FEATURE_COLS)
feature_data = [{"feature": k, "importance": round(float(v), 4)}
                for k, v in feat_imp.sort_values(ascending=False).head(10).items()]

# 5. Combine everything
output = {
    "meta": {
        "companies":   int(df["ticker"].nunique()),
        "years_range": [int(df["year"].min()), int(df["year"].max())],
        "sectors":     df["sector"].unique().tolist()
    },
    "records":            records,
    "sector_avg":         sector_data,
    "top_risk":           top_risk,
    "feature_importance": feature_data
}

# Save to both locations
react_data_dir = os.path.join(BASE_DIR, "dashboard", "src", "data")
os.makedirs(react_data_dir, exist_ok=True)

for path in [
    os.path.join(BASE_DIR, "data", "processed", "dashboard_export.json"),
    os.path.join(react_data_dir, "dashboard_export.json")
]:
    with open(path, "w") as f:
        json.dump(output, f, indent=2, default=str)
    print(f"Saved to {path}")

print(f"\nExport summary:")
print(f"  Total records:  {len(records)}")
print(f"  Companies:      {output['meta']['companies']}")
print(f"  Years:          {output['meta']['years_range']}")
print(f"  Top at-risk:    {[d['company_name'] for d in top_risk[:3]]}")

Saved to C:\Users\user\OneDrive\Desktop\Health-engine\financial-health-engine\data\processed\dashboard_export.json
Saved to C:\Users\user\OneDrive\Desktop\Health-engine\financial-health-engine\dashboard\src\data\dashboard_export.json

Export summary:
  Total records:  60
  Companies:      15
  Years:          [2021, 2025]
  Top at-risk:    ['Pick n Pay', 'MTN Group', 'Woolworths']
